In [21]:
from dotenv import load_dotenv

load_dotenv()

True

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate, FewShotPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.1,
)

In [23]:
examples = [
    {
        "movie": "Top Gun",
        "answer": "🛩️👨‍✈️🔥",
    },
    {
        "movie": "The Godfather",
        "answer": "👨‍👨‍👦🔫🍝",
    },
    {
        "movie": "Titanic",
        "answer": "🚢🌊💔",
    },
    {
        "movie": "Jurassic Park",
        "answer": "🦖🏝️🚙",
    },
    {
        "movie": "Harry Potter",
        "answer": "🧙‍♂️⚡🏰",
    },
]

In [24]:
example_prompt = PromptTemplate.from_template(
    "Movie: {movie}\nAnswer: {answer}"
)

fewshot_prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    prefix="Here are examples. These are only formatting examples, not conversation history.",
    suffix="",
    input_variables=[],
)

fewshot_examples = fewshot_prompt.format()

In [25]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [26]:
system_message = """
You are a movie emoji expert.

The examples below are only formatting examples.
They are not conversation history.
Never use the examples to answer questions about previous messages.

""" + fewshot_examples + """

Rules:
1. If the user's latest message is a movie title, reply with exactly three emojis.
2. Do not write words, explanations, punctuation, or extra text when replying with emojis.
3. If the user asks about conversation history, memory, or what movie was asked first, second, or last:
   - Do not reply with emojis.
   - Use only the actual previous messages from the conversation history.
   - If asked what the first movie was, answer with only the movie title.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

In [27]:
chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

/Users/browndays/VSCode/langchain/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
config = {"configurable": {"session_id": "movie_emoji_session"}}

response1 = chain_with_memory.invoke(
    {"question": "The Matrix"},
    config=config,
)

print("The Matrix:", response1.content)

response2 = chain_with_memory.invoke(
    {"question": "Finding Nemo"},
    config=config,
)

print("Finding Nemo:", response2.content)

The Matrix: 💊💻🕶️
Finding Nemo: 🐠🐢🐋


In [29]:
response3 = chain_with_memory.invoke(
    {"question": "what was the first movie I asked you about?"},
    config=config,
)

print(response3.content)

The Matrix
